# 🗄️ Data Warehouse — Analisis Tren Ekonomi Digital ASEAN 2019–2023
**Kelompok 9 | Kelas 2024-D | S1 Sains Data UNESA**

Pipeline: API Extraction → EDA Awal → Transformasi → Star Schema → OLAP (Atoti)

---
## 📦 0. Install & Import

In [ ]:
!pip install atoti requests pandas numpy matplotlib seaborn -q

In [ ]:
import requests, json, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
warnings.filterwarnings('ignore')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/staging', exist_ok=True)
os.makedirs('data/output', exist_ok=True)
print('Setup selesai ✓')

---
## 🌐 1. Ekstraksi Data — World Bank Data360 API (UNCTAD)

Sumber: https://data360api.worldbank.org — data ekspor & impor jasa digital dari UNCTADstat.
Tidak butuh API key, akses publik.

In [ ]:
BASE_URL = 'https://data360api.worldbank.org/data360/data'

ASEAN_COUNTRIES = {
    'BRN': 'Brunei Darussalam',
    'KHM': 'Cambodia',
    'IDN': 'Indonesia',
    'LAO': 'Lao PDR',
    'MYS': 'Malaysia',
    'MMR': 'Myanmar',
    'PHL': 'Philippines',
    'SGP': 'Singapore',
    'THA': 'Thailand',
    'VNM': 'Viet Nam'
}

INDICATORS = {
    'DP_TSD_EXPS': 'Exports of digitally deliverable services',
    'DP_TSD_IMPS': 'Imports of digitally deliverable services',
}

YEARS = list(range(2019, 2024))
print(f'Negara    : {list(ASEAN_COUNTRIES.keys())}')
print(f'Indikator : {list(INDICATORS.keys())}')
print(f'Tahun     : {YEARS}')

In [ ]:
# ── CEK RESPONS API MENTAH dulu sebelum tarik semua ──────────────────────
print('=== CEK RESPONS API — SGP, DP_TSD_EXPS, 2023 ===')

params_test = {
    'indicatorCode': 'DP_TSD_EXPS',
    'ref_area': 'SGP',
    'time_period': '2023',
    'format': 'json'
}

resp = requests.get(BASE_URL, params=params_test, timeout=30)
print(f'Status code : {resp.status_code}')
print(f'URL hit     : {resp.url}')

raw_test = resp.json()
print(f'\nTop-level keys: {list(raw_test.keys())}')

if 'data' in raw_test and len(raw_test['data']) > 0:
    print(f'\nJumlah record: {len(raw_test["data"])}')
    print(f'\nContoh 1 record (semua field yang tersedia):')
    print(json.dumps(raw_test['data'][0], indent=2))
    
    # Lihat semua kolom
    sample = pd.json_normalize(raw_test['data'])
    print(f'\nKolom API: {list(sample.columns)}')
else:
    print('Response kosong atau format berbeda:', raw_test)

In [ ]:
# ── Tarik semua data untuk 10 negara × 2 indikator × 5 tahun ─────────────
def fetch_all():
    all_records = []
    for ind_code, ind_label in INDICATORS.items():
        for ctry_code, ctry_name in ASEAN_COUNTRIES.items():
            print(f'  Fetching {ind_code} | {ctry_code} ({ctry_name})...', end=' ')
            count = 0
            for year in YEARS:
                params = {
                    'indicatorCode': ind_code,
                    'ref_area': ctry_code,
                    'time_period': str(year),
                    'format': 'json'
                }
                try:
                    r = requests.get(BASE_URL, params=params, timeout=30)
                    r.raise_for_status()
                    data = r.json()
                    if 'data' in data:
                        all_records.extend(data['data'])
                        count += len(data['data'])
                except Exception as e:
                    pass  # record tidak ada untuk kombinasi ini
            print(f'{count} records')
    return all_records

print('Mulai fetching...')
all_records = fetch_all()

print(f'\nTotal records dari API: {len(all_records)}')

# Simpan raw JSON
with open('data/raw/raw_unctad.json', 'w') as f:
    json.dump(all_records, f, indent=2)
print('Raw JSON tersimpan: data/raw/raw_unctad.json')

df_raw = pd.json_normalize(all_records)
print(f'Shape DataFrame mentah: {df_raw.shape}')
df_raw.head()

---
## 🔍 2. Eksplorasi Awal Data Mentah (Sebelum Transformasi)

Tujuan: pahami struktur, kelengkapan, sparsity, distribusi, dan outlier **sebelum** masuk pipeline ETL.

In [ ]:
# ── 2.1 Struktur & tipe data ─────────────────────────────────────────────
print('=== STRUKTUR DATA MENTAH ===')
print(f'Shape  : {df_raw.shape}')
print(f'Kolom  : {list(df_raw.columns)}')
print()
df_raw.info()
df_raw.head()

In [ ]:
# ── Standarisasi nama kolom ───────────────────────────────────────────────
df_raw.columns = [c.strip().upper() for c in df_raw.columns]

# Petakan ke nama standar (cek dan sesuaikan jika perlu)
rename_map = {}
for c in df_raw.columns:
    cl = c.upper()
    if 'REF_AREA' == cl:              rename_map[c] = 'REF_AREA'
    elif 'REF_AREA_LABEL' == cl:      rename_map[c] = 'REF_AREA_LABEL'
    elif cl in ('INDICATOR','IND'):    rename_map[c] = 'INDICATOR'
    elif 'INDICATOR_LABEL' in cl:      rename_map[c] = 'INDICATOR_LABEL'
    elif 'TIME_PERIOD' in cl or 'YEAR' == cl: rename_map[c] = 'TIME_PERIOD'
    elif cl == 'OBS_VALUE':            rename_map[c] = 'OBS_VALUE'
    elif cl == 'OBS_STATUS':           rename_map[c] = 'OBS_STATUS'
    elif 'UNIT_MEASURE' in cl:         rename_map[c] = 'UNIT_MEASURE'
    elif 'AGG_METHOD' in cl:           rename_map[c] = 'AGG_METHOD'

df_raw = df_raw.rename(columns=rename_map)

# Konversi OBS_VALUE ke numerik
if 'OBS_VALUE' in df_raw.columns:
    df_raw['OBS_VALUE'] = pd.to_numeric(df_raw['OBS_VALUE'], errors='coerce')

print('Kolom setelah standarisasi:', list(df_raw.columns))
print()
print('Preview:')
df_raw.head()

In [ ]:
# ── 2.2 Missing values per kolom ─────────────────────────────────────────
print('=== MISSING VALUES PER KOLOM ===')
missing_info = pd.DataFrame({
    'Missing': df_raw.isnull().sum(),
    'Persen (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
}).sort_values('Missing', ascending=False)
print(missing_info[missing_info['Missing'] > 0].to_string())

In [ ]:
# ── 2.3 Analisis Sparsity — kombinasi ideal vs aktual ────────────────────
print('=== ANALISIS SPARSITY ===')

n_countries  = len(ASEAN_COUNTRIES)
n_indicators = len(INDICATORS)
n_years      = len(YEARS)

kombinasi_ideal  = n_countries * n_indicators * n_years
kombinasi_aktual = len(df_raw)
sparsity_baris   = (1 - kombinasi_aktual / kombinasi_ideal) * 100

print(f'Negara                   : {n_countries}')
print(f'Indikator                : {n_indicators}')
print(f'Tahun                    : {n_years}')
print(f'Kombinasi ideal          : {kombinasi_ideal}')
print(f'Kombinasi dari API       : {kombinasi_aktual}')
print(f'Sparsity (record hilang) : {sparsity_baris:.1f}%')

if 'OBS_VALUE' in df_raw.columns:
    n_miss_val   = df_raw['OBS_VALUE'].isna().sum()
    pct_miss_val = n_miss_val / len(df_raw) * 100 if len(df_raw) > 0 else 0
    print(f'\nMissing OBS_VALUE        : {n_miss_val} / {len(df_raw)} ({pct_miss_val:.1f}%)')
    
    # Total sparsity (gabungan)
    total_sparsity = (1 - (kombinasi_aktual - n_miss_val) / kombinasi_ideal) * 100
    print(f'Total sparsity (overall) : {total_sparsity:.1f}%')

In [ ]:
# ── 2.4 Heatmap ketersediaan data per negara × tahun ─────────────────────
if 'REF_AREA' in df_raw.columns and 'TIME_PERIOD' in df_raw.columns:
    pivot_avail = df_raw.pivot_table(
        index='REF_AREA', columns='TIME_PERIOD',
        values='OBS_VALUE', aggfunc='count'
    )
    # Isi negara/tahun yang tidak muncul sama sekali dari API
    for yr in YEARS:
        if yr not in pivot_avail.columns:
            pivot_avail[yr] = 0
    for code in ASEAN_COUNTRIES:
        if code not in pivot_avail.index:
            pivot_avail.loc[code] = 0
    pivot_avail = pivot_avail[sorted(pivot_avail.columns)].fillna(0).astype(int)
    pivot_avail.index = [ASEAN_COUNTRIES.get(c, c) for c in pivot_avail.index]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot_avail, annot=True, fmt='d', cmap='YlGnBu',
                linewidths=0.5, ax=ax, vmin=0,
                cbar_kws={'label': 'Jumlah record tersedia'})
    ax.set_title('Ketersediaan Data per Negara & Tahun (dari API)', fontsize=12, pad=10)
    ax.set_xlabel('Tahun'); ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig('data/output/eda_01_sparsity_heatmap.png', dpi=150)
    plt.show()
    print('\nTabel ketersediaan:')
    print(pivot_avail.to_string())

In [ ]:
# ── 2.5 Missing OBS_VALUE per negara ─────────────────────────────────────
if 'REF_AREA_LABEL' in df_raw.columns:
    label_col = 'REF_AREA_LABEL'
elif 'REF_AREA' in df_raw.columns:
    label_col = 'REF_AREA'

miss_neg = (df_raw.groupby(label_col)['OBS_VALUE']
                  .apply(lambda x: x.isna().mean() * 100)
                  .round(1).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = ['#E24B4A' if v > 50 else '#378ADD' for v in miss_neg.values]
miss_neg.plot(kind='bar', ax=ax, color=colors_bar, edgecolor='none')
ax.set_title('Persentase Missing OBS_VALUE per Negara ASEAN (%)', fontsize=12)
ax.set_xlabel(''); ax.set_ylabel('Missing (%)')
ax.axhline(50, color='orange', linestyle='--', linewidth=1, label='50% threshold')
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('data/output/eda_02_missing_per_negara.png', dpi=150)
plt.show()

print('\nMissing % per negara:')
print(miss_neg.to_string())

In [ ]:
# ── 2.6 Distribusi OBS_VALUE: histogram & boxplot ────────────────────────
if 'OBS_VALUE' in df_raw.columns:
    df_nonnull = df_raw.dropna(subset=['OBS_VALUE'])
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram (semua data)
    axes[0].hist(df_nonnull['OBS_VALUE'], bins=20, color='#378ADD', edgecolor='white')
    axes[0].set_title('Distribusi OBS_VALUE (all countries)', fontsize=11)
    axes[0].set_xlabel('Nilai (Juta USD)'); axes[0].set_ylabel('Frekuensi')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    
    # Histogram tanpa Singapura (biar outlier terlihat jelas)
    if 'REF_AREA' in df_nonnull.columns:
        df_no_sgp = df_nonnull[df_nonnull['REF_AREA'] != 'SGP']
        axes[1].hist(df_no_sgp['OBS_VALUE'], bins=20, color='#D85A30', edgecolor='white')
        axes[1].set_title('Distribusi OBS_VALUE (tanpa Singapura)', fontsize=11)
        axes[1].set_xlabel('Nilai (Juta USD)'); axes[1].set_ylabel('Frekuensi')
        axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    
    plt.tight_layout()
    plt.savefig('data/output/eda_03_distribusi.png', dpi=150)
    plt.show()
    
    print('\nStatistik deskriptif OBS_VALUE:')
    print(df_nonnull['OBS_VALUE'].describe().round(2).to_string())
    
    print('\nSkewness :', round(df_nonnull['OBS_VALUE'].skew(), 3))
    print('Kurtosis  :', round(df_nonnull['OBS_VALUE'].kurt(), 3))

In [ ]:
# ── 2.7 Outlier: Boxplot per negara ──────────────────────────────────────
if 'OBS_VALUE' in df_raw.columns and 'REF_AREA_LABEL' in df_raw.columns:
    df_nonnull = df_raw.dropna(subset=['OBS_VALUE'])
    
    fig, ax = plt.subplots(figsize=(10, 4))
    df_nonnull.boxplot(column='OBS_VALUE', by='REF_AREA_LABEL',
                       ax=ax, showfliers=True,
                       boxprops=dict(color='#185FA5'),
                       flierprops=dict(marker='o', color='#E24B4A', markersize=5))
    ax.set_title('Boxplot OBS_VALUE per Negara (outlier terlihat)', fontsize=11)
    ax.set_xlabel(''); ax.set_ylabel('Nilai (Juta USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    plt.suptitle(''); plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('data/output/eda_04_outlier_boxplot.png', dpi=150)
    plt.show()
    
    print('\nRingkasan nilai per negara:')
    print(df_nonnull.groupby('REF_AREA_LABEL')['OBS_VALUE']
                    .agg(['count','min','mean','median','max'])
                    .round(2).sort_values('max', ascending=False)
                    .to_string())

In [ ]:
# ── 2.8 Ringkasan temuan EDA — untuk laporan ─────────────────────────────
print('=' * 55)
print('RINGKASAN TEMUAN EDA AWAL')
print('=' * 55)

if 'OBS_VALUE' in df_raw.columns:
    n_r = len(df_raw)
    n_miss = df_raw['OBS_VALUE'].isna().sum()
    sp_baris = (1 - n_r / kombinasi_ideal) * 100
    sp_val   = n_miss / n_r * 100

    print(f'  Total kombinasi ideal    : {kombinasi_ideal}')
    print(f'  Record dari API          : {n_r}')
    print(f'  Sparsity (baris)         : {sp_baris:.1f}%')
    print(f'  Missing OBS_VALUE        : {n_miss} ({sp_val:.1f}%)')
    print(f'  Skewness distribusi      : {df_raw["OBS_VALUE"].skew():.2f} (right-skewed)')
    
    if 'REF_AREA_LABEL' in df_raw.columns:
        max_country = (df_raw.dropna(subset=['OBS_VALUE'])
                             .groupby('REF_AREA_LABEL')['OBS_VALUE'].max()
                             .idxmax())
        print(f'  Negara nilai tertinggi   : {max_country}')
        
    print()
    print('  Temuan utama:')
    print('  - Data sangat sparse, banyak kombinasi negara-tahun-indikator tidak tersedia')
    print('  - Distribusi right-skewed ekstrem karena Singapura mendominasi')
    print('  - Beberapa negara (Myanmar, Laos, Thailand) data sangat terbatas')
    print('  - Tidak ada data negatif atau nonsensical')
print('=' * 55)

---
## 🔧 3. Transformasi — Bangun Star Schema

In [ ]:
INDICATOR_META = {
    'DP_TSD_EXPS': {
        'label'       : 'Exports of digitally deliverable services',
        'unit_measure': 'USD', 'unit_type': 'Currency',
        'unit_mult'   : 'Millions', 'agg_method': 'SUM'
    },
    'DP_TSD_IMPS': {
        'label'       : 'Imports of digitally deliverable services',
        'unit_measure': 'USD', 'unit_type': 'Currency',
        'unit_mult'   : 'Millions', 'agg_method': 'SUM'
    },
}

COUNTRY_LABELS = {
    'BRN':'Brunei Darussalam','KHM':'Cambodia','IDN':'Indonesia',
    'LAO':'Lao PDR','MYS':'Malaysia','MMR':'Myanmar',
    'PHL':'Philippines','SGP':'Singapore','THA':'Thailand','VNM':'Viet Nam'
}

df = df_raw.copy()

# Filter
df = df[df['REF_AREA'].isin(ASEAN_COUNTRIES.keys())]
df = df[df['TIME_PERIOD'].astype(str).isin([str(y) for y in YEARS])]
df['TIME_PERIOD'] = df['TIME_PERIOD'].astype(int)

# Hapus duplikat
df = df.drop_duplicates(subset=['REF_AREA','INDICATOR','TIME_PERIOD'])

# Bersihkan OBS_STATUS
df['OBS_STATUS'] = (df['OBS_STATUS']
                    .replace(['Not Available','N/A','','nan'], np.nan)
                    .fillna('A'))

# Tambah atribut dimensi
df['REF_AREA_LABEL']  = df['REF_AREA'].map(COUNTRY_LABELS)
df['KAWASAN']         = 'ASEAN'
df['PERIODE']         = '2019-2023'
df['INDICATOR_LABEL'] = df['INDICATOR'].map({k:v['label'] for k,v in INDICATOR_META.items()})
df['UNIT_MEASURE']    = df['INDICATOR'].map({k:v['unit_measure'] for k,v in INDICATOR_META.items()})
df['UNIT_TYPE']       = df['INDICATOR'].map({k:v['unit_type'] for k,v in INDICATOR_META.items()})
df['UNIT_MULT']       = df['INDICATOR'].map({k:v['unit_mult'] for k,v in INDICATOR_META.items()})
df['AGG_METHOD']      = df['INDICATOR'].map({k:v['agg_method'] for k,v in INDICATOR_META.items()})

print(f'Shape setelah transform: {df.shape}')
df.head()

In [ ]:
# Bangun tabel dimensi & fakta

# DIM_negara
dim_negara = (df[['REF_AREA','REF_AREA_LABEL','KAWASAN']]
              .drop_duplicates().reset_index(drop=True))
dim_negara.index += 1
dim_negara = dim_negara.reset_index().rename(columns={'index':'id_negara'})

# DIM_waktu
dim_waktu = pd.DataFrame({'TIME_PERIOD':list(range(2019,2024)),'PERIODE':['2019-2023']*5})
dim_waktu.index += 1
dim_waktu = dim_waktu.reset_index().rename(columns={'index':'id_waktu'})

# DIM_indikator
dim_indikator = (df[['INDICATOR','INDICATOR_LABEL','UNIT_MEASURE',
                      'UNIT_TYPE','UNIT_MULT','AGG_METHOD']]
                 .drop_duplicates().reset_index(drop=True))
dim_indikator.index += 1
dim_indikator = dim_indikator.reset_index().rename(columns={'index':'id_indikator'})

# FACT
fact = (df.merge(dim_negara[['id_negara','REF_AREA']], on='REF_AREA')
          .merge(dim_waktu[['id_waktu','TIME_PERIOD']], on='TIME_PERIOD')
          .merge(dim_indikator[['id_indikator','INDICATOR']], on='INDICATOR'))
fact = fact[['id_negara','id_waktu','id_indikator','OBS_VALUE','OBS_STATUS']].copy()
fact.index += 1
fact = fact.reset_index().rename(columns={'index':'id_perdagangan'})

# Simpan
dim_negara.to_csv('data/staging/dim_negara.csv', index=False)
dim_waktu.to_csv('data/staging/dim_waktu.csv', index=False)
dim_indikator.to_csv('data/staging/dim_indikator.csv', index=False)
fact.to_csv('data/staging/fact_perdagangan.csv', index=False)

print('Star schema tersimpan di data/staging/ ✓')
print(f'  dim_negara    : {len(dim_negara)} rows')
print(f'  dim_waktu     : {len(dim_waktu)} rows')
print(f'  dim_indikator : {len(dim_indikator)} rows')
print(f'  fact          : {len(fact)} rows')
print(f'  Sparsity fact : {fact["OBS_VALUE"].isna().mean()*100:.1f}%')
print()
print('DIM_negara:');    print(dim_negara.to_string(index=False))
print()
print('DIM_indikator:'); print(dim_indikator.to_string(index=False))

---
## 📊 4. Visualisasi untuk Laporan Kemajuan

In [ ]:
# Join semua tabel
df_join = (fact
           .merge(dim_negara,    on='id_negara')
           .merge(dim_waktu,     on='id_waktu')
           .merge(dim_indikator, on='id_indikator'))
df_join.columns = [c.lower() for c in df_join.columns]

df_val = df_join.dropna(subset=['obs_value']).copy()
eksp   = df_val[df_val['indicator'].str.contains('EXPS', na=False)]
imp    = df_val[df_val['indicator'].str.contains('IMPS', na=False)]

print(f'Data join shape (non-null): {df_val.shape}')
df_val.head()

In [ ]:
# Tren ekspor per negara
pivot_e = eksp.pivot_table(index='time_period', columns='ref_area_label', values='obs_value', aggfunc='sum')

fig, ax = plt.subplots(figsize=(10,5))
pivot_e.plot(ax=ax, marker='o')
ax.set_title('Tren Ekspor Jasa Digital ASEAN 2019–2023', fontsize=13)
ax.set_xlabel('Tahun'); ax.set_ylabel('Nilai (Juta USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('data/output/olap_01_tren_ekspor.png', dpi=150); plt.show()

In [ ]:
# Tren impor per negara
pivot_i = imp.pivot_table(index='time_period', columns='ref_area_label', values='obs_value', aggfunc='sum')

fig, ax = plt.subplots(figsize=(10,5))
pivot_i.plot(ax=ax, marker='s', linestyle='--')
ax.set_title('Tren Impor Jasa Digital ASEAN 2019–2023', fontsize=13)
ax.set_xlabel('Tahun'); ax.set_ylabel('Nilai (Juta USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('data/output/olap_02_tren_impor.png', dpi=150); plt.show()

In [ ]:
# Bar chart — total ekspor kumulatif per negara
total_e = eksp.groupby('ref_area_label')['obs_value'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9,5))
colors = ['#E24B4A' if i == 0 else '#378ADD' for i in range(len(total_e))]
total_e.plot(kind='bar', ax=ax, color=colors, edgecolor='none')
ax.set_title('Total Ekspor Jasa Digital Kumulatif 2019–2023', fontsize=12)
ax.set_xlabel(''); ax.set_ylabel('Nilai (Juta USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('data/output/olap_03_bar_ekspor.png', dpi=150); plt.show()

In [ ]:
# Surplus / Defisit per negara
grp_e = eksp.groupby(['ref_area_label','time_period'])['obs_value'].sum().reset_index(name='ekspor')
grp_i = imp.groupby(['ref_area_label','time_period'])['obs_value'].sum().reset_index(name='impor')
sd = grp_e.merge(grp_i, on=['ref_area_label','time_period'], how='outer').fillna(0)
sd['surplus'] = sd['ekspor'] - sd['impor']
total_sd = sd.groupby('ref_area_label')['surplus'].sum().sort_values()

colors_sd = ['#0F6E56' if v >= 0 else '#E24B4A' for v in total_sd.values]
fig, ax = plt.subplots(figsize=(9,5))
total_sd.plot(kind='barh', ax=ax, color=colors_sd, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Surplus / Defisit Jasa Digital Kumulatif 2019–2023', fontsize=12)
ax.set_xlabel('Nilai (Juta USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('data/output/olap_04_surplus_defisit.png', dpi=150); plt.show()

print('\nTabel Surplus / Defisit:')
res_sd = total_sd.reset_index()
res_sd.columns = ['Negara','Net (Juta USD)']
res_sd['Status'] = res_sd['Net (Juta USD)'].apply(lambda x: 'SURPLUS' if x>=0 else 'DEFISIT')
print(res_sd.sort_values('Net (Juta USD)', ascending=False).to_string(index=False))

In [ ]:
# Total kawasan ASEAN per tahun
t_e = eksp.groupby('time_period')['obs_value'].sum()
t_i = imp.groupby('time_period')['obs_value'].sum()

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(t_e.index, t_e.values, marker='o', label='Ekspor', color='#185FA5')
ax.plot(t_i.index, t_i.values, marker='s', label='Impor',  color='#D85A30', linestyle='--')
ax.fill_between(t_e.index, t_e.values, t_i.values, alpha=0.08, color='#378ADD')
ax.set_title('Total Ekspor vs Impor Jasa Digital Kawasan ASEAN per Tahun', fontsize=12)
ax.set_xlabel('Tahun'); ax.set_ylabel('Nilai (Juta USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('data/output/olap_05_total_kawasan.png', dpi=150); plt.show()

---
## 🧊 5. OLAP dengan Atoti — CUBE Queries

In [ ]:
import atoti as tt

session = tt.Session.local()

df_cube = df_join.dropna(subset=['obs_value']).copy()
df_cube['time_period'] = df_cube['time_period'].astype(int)
df_cube['obs_value']   = pd.to_numeric(df_cube['obs_value'], errors='coerce')

store = session.read_pandas(
    df_cube,
    store_name='perdagangan_jasa_digital',
    keys=['id_perdagangan']
)

cube = session.create_cube(store, 'ASEAN Digital Trade Cube')
l, m = cube.levels, cube.measures

# Hierarki geografis & waktu
l['kawasan']           = store['kawasan']
l['negara']            = store['ref_area_label']
l['tahun']             = store['time_period']
l['jenis_perdagangan'] = store['indicator_label']

# Measures
m['Nilai (Juta USD)'] = tt.agg.sum(
    store['obs_value'],
    scope=tt.scope.origin(l['jenis_perdagangan'])
)
m['Jumlah Negara'] = tt.agg.count_distinct(store['ref_area_label'])

print('Cube berhasil dibuat ✓')
print(f'Atoti UI: {session.url}')

In [ ]:
# CUBE QUERY 1: Ekspor & Impor per Negara (kumulatif 2019-2023)
print('=== QUERY 1: Ekspor & Impor per Negara ===')
q1 = cube.query(
    m['Nilai (Juta USD)'],
    levels=[l['negara'], l['jenis_perdagangan']]
)
print(q1.to_string())

In [ ]:
# CUBE QUERY 2: Tren ekspor per tahun per negara
print('=== QUERY 2: Tren Ekspor per Tahun per Negara ===')
q2 = cube.query(
    m['Nilai (Juta USD)'],
    levels=[l['negara'], l['tahun']],
    condition=l['jenis_perdagangan'] == 'Exports of digitally deliverable services'
)
print(q2.to_string())

In [ ]:
# CUBE QUERY 3: Surplus / Defisit
print('=== QUERY 3: Surplus / Defisit per Negara ===')
q3 = cube.query(m['Nilai (Juta USD)'], levels=[l['negara'], l['jenis_perdagangan']])
if isinstance(q3, pd.DataFrame):
    pivot3 = q3.reset_index().pivot_table(
        index='negara', columns='jenis_perdagangan',
        values='Nilai (Juta USD)', aggfunc='sum'
    )
    ek = [c for c in pivot3.columns if 'Export' in c]
    im = [c for c in pivot3.columns if 'Import' in c]
    if ek and im:
        pivot3['Surplus/Defisit'] = pivot3[ek[0]].fillna(0) - pivot3[im[0]].fillna(0)
        pivot3['Status'] = pivot3['Surplus/Defisit'].apply(lambda x: 'SURPLUS' if x >= 0 else 'DEFISIT')
        print(pivot3[['Surplus/Defisit','Status']].sort_values('Surplus/Defisit', ascending=False).to_string())

In [ ]:
# CUBE QUERY 4: Total kawasan ASEAN per tahun
print('=== QUERY 4: Total Kawasan ASEAN per Tahun ===')
q4 = cube.query(
    m['Nilai (Juta USD)'],
    levels=[l['tahun'], l['jenis_perdagangan']]
)
print(q4.to_string())

In [ ]:
# Buka dashboard interaktif Atoti di Jupyter/Colab
session.visualize()

---
## ✅ Selesai — Download Semua Output

In [ ]:
import shutil
shutil.make_archive('dwh_output', 'zip', 'data/output')
print('Semua output di-zip ke dwh_output.zip ✓')
print('Download dari panel file di kiri layar Colab.')